# 7. IA Generativa y LLMs II
## Del reporte corporativo a la valoración de empresas

### De frases sueltas a documentos completos

En la sesión anterior clasificamos el sentimiento de **frases cortas de noticias**. Un analista de
valoración, sin embargo, no trabaja con titulares: trabaja con **reportes anuales de 100 a 300 páginas**
(el 10-K en EE. UU., el informe de gestión o el informe periódico de fin de ejercicio en Colombia).
Ahí están los insumos de un DCF: cuánto vendió la empresa, qué margen obtuvo, cuánto invirtió, qué
espera la gerencia para el próximo año y qué riesgos podrían descarrilar ese plan.

Leer ese documento con un LLM trae problemas nuevos que con frases cortas no existían:

- **No cabe entero en el modelo** (ventana de contexto) → hay que segmentarlo y *recuperar* solo lo relevante.
- **Las cifras importan al decimal** → toda cifra extraída debe **verificarse** contra una fuente estructurada.
- **El valor está en el cambio**: lo que la empresa *agregó* este año a sus factores de riesgo dice más que lo que repite.
- **El LLM no debe inventar el valor de la empresa**: su papel es leer y citar; los números del DCF los
  pone el analista con reglas explícitas.

### ¿Qué se pretende?

1. Descargar un reporte corporativo real (10-K de la SEC) y sus estados financieros estructurados (XBRL).
2. Segmentar el reporte en secciones (*Factores de riesgo* y *MD&A*) y medir su **tono** y su **cambio
   interanual** (la idea de *Lazy Prices*, Cohen, Malloy y Nguyen, 2020).
3. Construir un sistema **RAG** (recuperación + generación) que responda preguntas sobre el reporte
   **citando el fragmento** de donde sacó cada respuesta.
4. **Extraer** insumos de valoración en JSON y **verificarlos** contra los datos XBRL.
5. Usar la lectura del LLM para construir **escenarios de un DCF** con reglas transparentes y redactar una
   **nota de valoración asistida por IA** (insumo del Taller 2).

| Paso | Técnica | Pregunta de valoración que responde |
|---|---|---|
| 1 | API EDGAR + XBRL | ¿Qué dicen el reporte y los estados financieros? |
| 2 | Segmentación y *chunking* | ¿Dónde está cada tema dentro de 200 páginas? |
| 3 | FinBERT + similitud entre años | ¿El tono empeoró? ¿Qué riesgos son nuevos? |
| 4 | RAG con citas | ¿Qué guía dio la gerencia? ¿Por qué cayó el margen? |
| 5 | Extracción estructurada + verificación | ¿Las cifras que leyó el LLM son correctas? |
| 6 | DCF por escenarios | ¿Cuánto vale la empresa si el texto tiene razón? |

### 0.1 Entorno de ejecución

Igual que en la sesión 7: pensado para **Colab**. Las partes 1 a 3 corren en CPU; las partes con LLM
(4 a 6) necesitan **GPU T4** si usas el modelo local, o una **clave de API** si prefieres un modelo
externo (ver sección 0.3).

In [ ]:
# Mismas versiones fijadas de la sesión 7 (no reinstalamos torch: Colab ya trae la build correcta para su GPU)
!pip install -q \
    "transformers==4.46.3" \
    "accelerate==1.0.1" \
    "bitsandbytes==0.44.1" \
    "sentence-transformers==3.2.1" \
    "huggingface_hub==0.25.2" \
    "scikit-learn==1.5.2" \
    "pandas==2.2.3" \
    "matplotlib==3.9.2" \
    "seaborn==0.13.2" \
    "beautifulsoup4==4.12.3" \
    "pypdf==5.1.0"

In [ ]:
import os
import re
import json
import time
import random
import difflib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import torch
from bs4 import BeautifulSoup

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
pd.set_option("display.max_colwidth", 160)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | dispositivo: {DEVICE}")

### 0.2 Parámetros de la sesión

- `TICKER`: empresa a analizar. Funciona con cualquier empresa **no financiera** que reporte a la SEC
  (para bancos y aseguradoras el DCF por flujo de caja libre de la firma no aplica).
- `SEC_USER_AGENT`: la SEC **exige** que cada solicitud identifique a quien la hace (nombre y correo).
  Si no lo cambias, el cuaderno usa automáticamente el **plan B**.
- `FUENTE`: `"edgar"` (reporte real) o `"ficticio"` (un 10-K didáctico embebido en el cuaderno, que
  garantiza que la clase funcione aunque falle internet o la SEC cambie algo).

In [ ]:
TICKER = "AAPL"                                   # cámbialo por la empresa de tu proyecto integrador
SEC_USER_AGENT = "Nombre Apellido correo@udea.edu.co"   # ⚠️ pon tu nombre y correo reales
FUENTE = "edgar"                                 # "edgar" o "ficticio"

if "correo@udea.edu.co" in SEC_USER_AGENT and FUENTE == "edgar":
    print("⚠️  No configuraste SEC_USER_AGENT: usaremos el reporte ficticio (plan B).")
    FUENTE = "ficticio"

### 0.3 El LLM: local (Qwen) o vía API

Definimos **una sola función** `llamar_llm(mensajes)` que el resto del cuaderno usa sin saber qué modelo
hay detrás. Así puedes cambiar de modelo sin tocar nada más.

- `"local"`: `Qwen/Qwen2.5-3B-Instruct` en la GPU de Colab, igual que en la sesión 7. Gratis, pero un
  modelo de 3B se equivoca más en extracción de cifras (lo cual, como veremos, es una buena lección).
- `"api"`: cualquier proveedor con API compatible con OpenAI (OpenAI, Groq, Google Gemini, un servidor
  Ollama propio…). Guarda la clave en los **Secretos de Colab** (ícono de llave) con el nombre
  `LLM_API_KEY`; nunca la escribas en el cuaderno.

In [ ]:
BACKEND_LLM = "local"                       # "local" o "api"
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct" # modelo local (prueba "Qwen/Qwen2.5-7B-Instruct" si tienes GPU)
API_BASE_URL = "https://api.openai.com/v1"  # cambia según tu proveedor
API_MODEL = "gpt-4o-mini"                   # nombre del modelo en ese proveedor

if BACKEND_LLM == "local":
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    tokenizer_llm = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
    if tokenizer_llm.pad_token is None:
        tokenizer_llm.pad_token = tokenizer_llm.eos_token
    if DEVICE == "cuda":
        memoria_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        if memoria_gb < 10:
            cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                     bnb_4bit_quant_type="nf4")
            modelo_llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME, quantization_config=cfg,
                                                              device_map="auto")
        else:
            modelo_llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME, torch_dtype=torch.float16,
                                                              device_map="auto")
    else:
        print("⚠️  Sin GPU: el LLM local será muy lento. Considera BACKEND_LLM = 'api'.")
        modelo_llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME)
    modelo_llm.eval()
    modelo_llm.generation_config.do_sample = False
    modelo_llm.generation_config.temperature = None
    modelo_llm.generation_config.top_p = None
    modelo_llm.generation_config.top_k = None

    def llamar_llm(mensajes, max_new_tokens=400):
        prompt = tokenizer_llm.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
        entradas = tokenizer_llm(prompt, return_tensors="pt").to(modelo_llm.device)
        with torch.no_grad():
            salida = modelo_llm.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False,
                                         pad_token_id=tokenizer_llm.pad_token_id)
        return tokenizer_llm.decode(salida[0, entradas["input_ids"].shape[1]:], skip_special_tokens=True)

else:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("LLM_API_KEY")
    except Exception:
        API_KEY = os.environ.get("LLM_API_KEY")

    def llamar_llm(mensajes, max_new_tokens=400):
        r = requests.post(
            f"{API_BASE_URL}/chat/completions",
            headers={"Authorization": f"Bearer {API_KEY}"},
            json={"model": API_MODEL, "messages": mensajes, "temperature": 0, "max_tokens": max_new_tokens},
            timeout=120,
        )
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"]

print(f"LLM listo ({BACKEND_LLM}).")

Como en la sesión 7, le pediremos al LLM respuestas en **JSON**. Aquí los JSON serán más complejos (con
listas adentro), así que el parser busca el primer bloque `{...}` **balanceado** en lugar de usar una
expresión regular simple, y devuelve `None` si no logra leerlo.

In [ ]:
def extraer_json(texto):
    """Devuelve el primer objeto JSON balanceado que aparezca en `texto`, o None."""
    inicio = texto.find("{")
    while inicio != -1:
        profundidad, en_cadena, escape = 0, False, False
        for i in range(inicio, len(texto)):
            c = texto[i]
            if en_cadena:
                escape = (c == "\\") and not escape
                if c == '"' and not escape:
                    en_cadena = False
                continue
            if c == '"':
                en_cadena = True
            elif c == "{":
                profundidad += 1
            elif c == "}":
                profundidad -= 1
                if profundidad == 0:
                    try:
                        return json.loads(texto[inicio:i + 1])
                    except json.JSONDecodeError:
                        break
        inicio = texto.find("{", inicio + 1)
    return None


print(extraer_json('Claro, aquí está: {"a": 1, "b": [{"c": "x}"}]} ¡listo!'))

<a id="sec1"></a>
## 1. Datos: el reporte anual (10-K) y los estados financieros (XBRL)

### 1.1 Anatomía de un 10-K

El 10-K es el reporte anual obligatorio de las empresas listadas en EE. UU. Tiene una estructura fija
de *Items*; para valoración nos interesan sobre todo dos:

| Sección | Contenido | Uso en valoración |
|---|---|---|
| **Item 1A – Risk Factors** | Riesgos que la gerencia considera materiales | Prima de riesgo, escenarios pesimistas |
| **Item 7 – MD&A** (*Management's Discussion and Analysis*) | La gerencia explica resultados, márgenes, liquidez y perspectivas | Supuestos de crecimiento, márgenes, capex |
| Item 8 – Financial Statements | Estados financieros auditados | Cifras base (las tomaremos de XBRL) |

Además del texto, la SEC publica los estados financieros en **XBRL**: cada cifra viene etiquetada
(`Revenues`, `OperatingIncomeLoss`…) y se puede descargar como JSON. Esto nos da algo muy valioso: una
**"verdad de referencia"** para comprobar si el LLM leyó bien las cifras del texto.

> **¿Y en Colombia?** Los emisores de valores publican el *informe periódico de fin de ejercicio* y el
> *informe de gestión* (Superintendencia Financiera, RNVE). No hay una API equivalente a EDGAR: suelen
> ser PDF. En el ejercicio 2 adaptamos el mismo flujo a un PDF en español.

### 1.2 Plan B: un 10-K didáctico embebido

Antes de ir a la SEC dejamos listo un **reporte ficticio** de *Cordillera Outfitters Inc.* (empresa
inventada) con dos años de *Risk Factors* y *MD&A* escritos al estilo de un 10-K real, y sus cifras
financieras. Está diseñado para que aparezcan los fenómenos que queremos estudiar: un riesgo nuevo,
un margen que cae, una guía de crecimiento que no se cumplió.

In [ ]:
AVISO = ("[Illustrative document for teaching purposes: Cordillera Outfitters Inc. is a fictitious "
         "company and all names and figures are invented]. ")

REPORTE_FICTICIO = {
    2024: {
        "1A": AVISO + """ITEM 1A. RISK FACTORS. Risks Related to Our Business and Industry.
Changes in consumer discretionary spending could adversely affect our results. Our products are discretionary purchases. A deterioration in economic conditions, higher interest rates or inflation could reduce consumer demand for outdoor apparel and equipment, which could adversely affect our net sales and profitability.
We operate in a highly competitive industry. We compete with large global brands and with the private labels of our retail customers. Competitive pressure could force us to increase promotional activity or reduce prices, which could reduce our gross margin.
Our supply chain is concentrated in a limited number of countries. Substantially all of our products are manufactured by independent suppliers, primarily located in Vietnam and Bangladesh. Disruptions at these suppliers, including labor shortages or natural disasters, could delay deliveries and increase our costs.
Unseasonable weather could reduce demand for our products. A significant portion of our net sales occurs in the fall and winter seasons. Warmer than normal weather could reduce sales of outerwear and lead to excess inventory and markdowns.
A breach of our information technology systems could harm our business. We rely on information systems to operate our e-commerce platform and our distribution centers. A cybersecurity incident could disrupt operations, expose us to litigation and damage our reputation.
Fluctuations in foreign currency exchange rates could affect our results. Approximately 30% of our net sales are generated outside the United States. A stronger U.S. dollar reduces the dollar value of our international sales.""",
        "7": AVISO + """ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS.
Overview. Cordillera Outfitters designs, markets and distributes outdoor apparel, footwear and equipment through wholesale and direct-to-consumer channels.
Fiscal 2024 compared with fiscal 2023. Net sales increased 9.7% to $5,320 million in fiscal 2024 from $4,850 million in fiscal 2023, driven by strong demand for our outerwear collection and growth in our e-commerce business. Direct-to-consumer net sales increased 15% to $1,910 million, and wholesale net sales increased 6.9% to $3,410 million.
Gross margin expanded 60 basis points to 46.1%, reflecting lower freight costs and a favorable channel mix. Operating income increased 9.7% to $702 million, and operating margin was 13.2%, unchanged from the prior year. Our effective tax rate was 22.0% in fiscal 2024, compared with 21.5% in fiscal 2023.
Liquidity and capital resources. Cash flow from operations was $655 million. Capital expenditures were $240 million, primarily related to store openings and technology investments. Depreciation and amortization expense was $188 million. We repurchased $120 million of our common stock and paid dividends of $85 million. As of fiscal year-end we had cash and cash equivalents of $760 million and total debt of $1,150 million.
Outlook. For fiscal 2025, we expect net sales to grow in the high single digits and operating margin to expand modestly. We expect capital expenditures of approximately $300 million, including the construction of a new distribution center.""",
    },
    2025: {
        "1A": AVISO + """ITEM 1A. RISK FACTORS. Risks Related to Our Business and Industry.
Changes in consumer discretionary spending could adversely affect our results. Our products are discretionary purchases. A deterioration in economic conditions, higher interest rates or inflation could reduce consumer demand for outdoor apparel and equipment, which could adversely affect our net sales and profitability.
We operate in a highly competitive industry. We compete with large global brands and with the private labels of our retail customers. Competitive pressure could force us to increase promotional activity or reduce prices, which could reduce our gross margin.
Changes in trade policy, including new tariffs, could materially increase our costs. In fiscal 2025 the United States imposed additional tariffs on apparel imported from several countries where our suppliers operate. These tariffs increased our cost of goods sold and reduced our gross margin. Further increases in tariffs could materially and adversely affect our profitability, and we may not be able to pass these costs on to consumers.
Our supply chain is concentrated in a limited number of countries. Substantially all of our products are manufactured by independent suppliers, primarily located in Vietnam and Bangladesh. Disruptions at these suppliers, including labor shortages, port congestion or natural disasters, could delay deliveries and increase our costs. During fiscal 2025 freight costs increased significantly, and these pressures may continue.
We depend on a small number of wholesale customers. Our largest wholesale customer accounted for approximately 14% of our net sales in fiscal 2025. The loss of this customer, or a significant reduction in its orders, could adversely affect our results of operations.
Unseasonable weather could reduce demand for our products. A significant portion of our net sales occurs in the fall and winter seasons. Warmer than normal weather could reduce sales of outerwear and lead to excess inventory and markdowns.
A breach of our information technology systems could harm our business. We rely on information systems to operate our e-commerce platform and our distribution centers. A cybersecurity incident could disrupt operations, expose us to litigation and damage our reputation.
Fluctuations in foreign currency exchange rates could affect our results. Approximately 31% of our net sales are generated outside the United States. A stronger U.S. dollar reduces the dollar value of our international sales.""",
        "7": AVISO + """ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS.
Overview. Cordillera Outfitters designs, markets and distributes outdoor apparel, footwear and equipment through wholesale and direct-to-consumer channels.
Fiscal 2025 compared with fiscal 2024. Net sales increased 5.5% to $5,610 million in fiscal 2025 from $5,320 million in fiscal 2024. Growth was below our initial expectations, as softer wholesale demand in North America offset continued strength in direct-to-consumer. Direct-to-consumer net sales increased 12% to $2,140 million, while wholesale net sales increased 1.8% to $3,470 million.
Gross margin decreased 180 basis points to 44.3%, primarily due to new tariffs on imported products, higher freight costs and increased promotional activity. Operating income decreased 5.7% to $662 million, and operating margin declined to 11.8% from 13.2%. Our effective tax rate was 22.0%, consistent with fiscal 2024.
Liquidity and capital resources. Cash flow from operations was $610 million. Capital expenditures were $310 million, primarily related to the construction of our new distribution center in Nevada. Depreciation and amortization expense was $205 million. We repurchased $150 million of our common stock and paid dividends of $90 million. As of fiscal year-end we had cash and cash equivalents of $820 million and total debt of $1,150 million.
Outlook. For fiscal 2026, we expect net sales to grow in the low-to-mid single digits and operating margin to be flat to down 50 basis points compared with fiscal 2025, reflecting continued tariff headwinds partially offset by cost savings initiatives. We expect capital expenditures of approximately $260 million as the distribution center is completed.""",
    },
}

# Cifras "XBRL" del caso ficticio, en millones de USD (acciones en millones)
FINANCIEROS_FICTICIOS = pd.DataFrame(
    {
        "ingresos":                 [4850, 5320, 5610],
        "utilidad_operativa":       [640, 702, 662],
        "utilidad_antes_impuestos": [625, 690, 641],
        "impuestos":                [134, 152, 141],
        "dep_amort":                [172, 188, 205],
        "capex":                    [210, 240, 310],
        "caja":                     [550, 760, 820],
        "deuda":                    [1150, 1150, 1150],
        "acciones":                 [218, 214, 210],
    },
    index=pd.Index([2023, 2024, 2025], name="anio_fiscal"),
)
PRECIO_FICTICIO = 32.0   # precio de mercado inventado, USD por acción

### 1.3 Plan A: descarga desde EDGAR

Usamos tres servicios **gratuitos y sin registro** de la SEC (solo exigen el `User-Agent`):

1. `company_tickers.json` → convierte el ticker en el **CIK** (identificador de la empresa en la SEC).
2. `submissions/CIK##########.json` → lista de reportes presentados; filtramos los **10-K**.
3. `api/xbrl/companyfacts/CIK##########.json` → todas las cifras XBRL históricas de la empresa.

La SEC permite máximo 10 solicitudes por segundo; hacemos pausas cortas entre llamadas por cortesía.

In [ ]:
HEADERS_SEC = {"User-Agent": SEC_USER_AGENT}


def get_sec(url):
    time.sleep(0.2)
    r = requests.get(url, headers=HEADERS_SEC, timeout=60)
    r.raise_for_status()
    return r


def obtener_cik(ticker):
    for d in get_sec("https://www.sec.gov/files/company_tickers.json").json().values():
        if d["ticker"].upper() == ticker.upper():
            return int(d["cik_str"]), d["title"]
    raise ValueError(f"No encontré el ticker {ticker} en la SEC")


def listar_10k(cik, n=2):
    rec = get_sec(f"https://data.sec.gov/submissions/CIK{cik:010d}.json").json()["filings"]["recent"]
    df = pd.DataFrame({k: rec[k] for k in ["form", "accessionNumber", "filingDate", "reportDate", "primaryDocument"]})
    return df[df["form"] == "10-K"].head(n).reset_index(drop=True)


def normalizar_espacios(texto):
    return re.sub(r"\s+", " ", texto.replace("\xa0", " ")).strip()


def descargar_texto_10k(cik, accession, documento):
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession.replace('-', '')}/{documento}"
    soup = BeautifulSoup(get_sec(url).text, "html.parser")
    for etiqueta in soup(["script", "style", "ix:header"]):   # ix:header = metadatos XBRL ocultos
        etiqueta.decompose()
    return normalizar_espacios(soup.get_text(" "))

**Extraer las secciones.** Un 10-K menciona "Item 7. Management's Discussion…" varias veces: en el índice,
en referencias cruzadas ("ver Item 7") y en el título real de la sección. La regla que usamos: para cada
aparición del **título siguiente** (p. ej. "Item 7A" u "Item 8"), tomamos el título de inicio **más
cercano antes de él**, y nos quedamos con el tramo más largo. El índice produce tramos cortísimos y las
referencias cruzadas quedan descartadas porque siempre hay un título más cercano.

In [ ]:
SEP = r"\.?\s*[\-–—:]?\s*"
PATRONES_SECCION = {
    "1A": (rf"item\s*1a{SEP}risk\s+factors",
           rf"item\s*1b{SEP}unresolved|item\s*1c{SEP}cybersecurity|item\s*2{SEP}properties"),
    "7":  (rf"item\s*7{SEP}management",
           rf"item\s*7a{SEP}quantitative|item\s*8{SEP}financial\s+statements"),
}


def extraer_seccion(texto, clave):
    patron_ini, patron_fin = PATRONES_SECCION[clave]
    inicios = [m.start() for m in re.finditer(patron_ini, texto, flags=re.I)]
    tramos = {}   # inicio más cercano → primer título de cierre después de él
    for fin in re.finditer(patron_fin, texto, flags=re.I):
        previos = [s for s in inicios if s < fin.start()]
        if previos:
            tramos[previos[-1]] = min(tramos.get(previos[-1], fin.start()), fin.start())
    return max((texto[i:f] for i, f in tramos.items()), key=len, default="")

**Cifras XBRL.** Cada concepto contable puede aparecer con distintas etiquetas según la empresa y el año
(por ejemplo, los ingresos pueden estar como `Revenues` o como `RevenueFromContractWithCustomerExcludingAssessedTax`).
Por eso cada variable tiene una **lista de etiquetas candidatas**, y para cada año usamos la primera que
tenga dato. Nos quedamos con cifras **anuales** (duración ≈ 1 año) reportadas en formularios 10-K.

In [ ]:
ETIQUETAS_XBRL = {
    "ingresos": ["RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues", "SalesRevenueNet",
                 "RevenueFromContractWithCustomerIncludingAssessedTax"],
    "utilidad_operativa": ["OperatingIncomeLoss"],
    "utilidad_antes_impuestos": [
        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeLossFromEquityMethodInvestments"],
    "impuestos": ["IncomeTaxExpenseBenefit"],
    "dep_amort": ["DepreciationDepletionAndAmortization", "DepreciationAmortizationAndAccretionNet",
                  "DepreciationAndAmortization", "Depreciation"],
    "capex": ["PaymentsToAcquirePropertyPlantAndEquipment", "PaymentsToAcquireProductiveAssets"],
    "caja": ["CashAndCashEquivalentsAtCarryingValue",
             "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"],
    "deuda": ["LongTermDebt"],
    "deuda_lp": ["LongTermDebtNoncurrent"],
    "deuda_cp": ["LongTermDebtCurrent", "DebtCurrent"],
    "gasto_intereses": ["InterestExpense", "InterestExpenseNonoperating", "InterestExpenseDebt"],
    "acciones": ["WeightedAverageNumberOfDilutedSharesOutstanding"],
}
VARIABLES_SALDO = {"caja", "deuda", "deuda_lp", "deuda_cp"}   # saldos a una fecha (no flujos del año)


def serie_anual(facts, etiquetas, es_flujo=True):
    gaap = facts["facts"].get("us-gaap", {})
    combinada = None
    for etiqueta in etiquetas:
        if etiqueta not in gaap:
            continue
        unidades = gaap[etiqueta]["units"]
        obs = unidades.get("USD") or unidades.get("shares") or []
        df = pd.DataFrame([o for o in obs if str(o.get("form", "")).startswith("10-K")])
        if df.empty:
            continue
        df["end"] = pd.to_datetime(df["end"])
        if es_flujo:
            if "start" not in df:
                continue
            duracion = (df["end"] - pd.to_datetime(df["start"])).dt.days
            df = df[duracion.between(350, 380)]
        df = df.assign(anio_fiscal=df["end"].dt.year).sort_values("filed")
        serie = df.drop_duplicates("anio_fiscal", keep="last").set_index("anio_fiscal")["val"]
        combinada = serie if combinada is None else combinada.combine_first(serie)
    return combinada


def construir_financieros(facts, n_anios=3):
    columnas = {}
    for var, etiquetas in ETIQUETAS_XBRL.items():
        s = serie_anual(facts, etiquetas, es_flujo=var not in VARIABLES_SALDO)
        if s is not None:
            columnas[var] = s / 1e6        # a millones (USD o acciones)
    df = pd.DataFrame(columnas).sort_index().reindex(columns=list(ETIQUETAS_XBRL))
    # Deuda total: LongTermDebt si existe; si no, porción de largo plazo + porción corriente
    df["deuda"] = df["deuda"].combine_first(df["deuda_lp"].add(df["deuda_cp"], fill_value=0))
    # Algunas empresas no reportan "utilidad operativa": la aproximamos con utilidad antes de impuestos + intereses
    aprox = df["utilidad_antes_impuestos"] + df["gasto_intereses"].fillna(0)
    if df["utilidad_operativa"].isna().any():
        print("⚠️  Falta OperatingIncomeLoss en algunos años: se aproxima como utilidad antes de impuestos + intereses.")
    df["utilidad_operativa"] = df["utilidad_operativa"].combine_first(aprox)
    df = df.drop(columns=["deuda_lp", "deuda_cp", "gasto_intereses"]).dropna(subset=["ingresos", "utilidad_operativa"])
    return df.tail(n_anios)

In [ ]:
TEXTOS = {}          # {año_fiscal: {"1A": texto, "7": texto}}
EMPRESA = None

if FUENTE == "edgar":
    try:
        cik, EMPRESA = obtener_cik(TICKER)
        filings = listar_10k(cik, n=2)
        display(filings)
        for _, f in filings.iterrows():
            texto = descargar_texto_10k(cik, f["accessionNumber"], f["primaryDocument"])
            anio = int(f["reportDate"][:4])
            TEXTOS[anio] = {clave: extraer_seccion(texto, clave) for clave in PATRONES_SECCION}
            print(f"{anio}: 10-K con {len(texto.split()):,} palabras | "
                  f"1A: {len(TEXTOS[anio]['1A'].split()):,} | 7: {len(TEXTOS[anio]['7'].split()):,}")
        facts = get_sec(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json").json()
        FINANCIEROS = construir_financieros(facts)
        if any(len(s.split()) < 300 for t in TEXTOS.values() for s in t.values()):
            raise ValueError("Alguna sección quedó casi vacía: revisa PATRONES_SECCION para esta empresa")
    except Exception as error:
        print(f"Plan A falló → usamos el reporte ficticio.\n  {type(error).__name__}: {str(error)[:200]}")
        FUENTE = "ficticio"

if FUENTE == "ficticio":
    EMPRESA = "Cordillera Outfitters Inc. (ficticia)"
    TEXTOS = {anio: {k: normalizar_espacios(v) for k, v in secc.items()} for anio, secc in REPORTE_FICTICIO.items()}
    FINANCIEROS = FINANCIEROS_FICTICIOS.copy()

ANIOS = sorted(TEXTOS)
ANIO_ACTUAL, ANIO_PREVIO = ANIOS[-1], ANIOS[0]
print(f"\nEmpresa: {EMPRESA} | años de texto: {ANIOS}")
FINANCIEROS

**Diagnóstico de datos.** Cada empresa etiqueta su XBRL a su manera. Si alguna variable quedó vacía,
búscala en el 10-K (estado de flujos de caja o balance) y complétala a mano, por ejemplo
`FINANCIEROS.loc[2025, "dep_amort"] = 11_445`. Es mejor un dato manual verificado que un DCF con huecos.

In [ ]:
faltantes = FINANCIEROS.isna()
if faltantes.values.any():
    print("⚠️  Variables sin dato XBRL (complétalas a mano antes de la sección 6):")
    print(faltantes.stack()[lambda s: s].index.tolist())
else:
    print("✅ Todas las variables tienen dato para los años analizados.")

**Precio de mercado.** Para comparar el valor estimado con el precio usamos `yfinance` si está
disponible; si falla, escribe el precio a mano en `PRECIO_MERCADO`.

In [ ]:
PRECIO_MERCADO = None
if FUENTE == "ficticio":
    PRECIO_MERCADO = PRECIO_FICTICIO
else:
    try:
        import yfinance as yf
        PRECIO_MERCADO = float(yf.Ticker(TICKER).history(period="5d")["Close"].iloc[-1])
    except Exception as error:
        print(f"No pude descargar el precio ({error}). Escríbelo a mano en PRECIO_MERCADO.")
print(f"Precio de mercado: {PRECIO_MERCADO}")

<a id="sec2"></a>
## 2. Segmentación: oraciones y *chunks*

Un 10-K real tiene entre 30 000 y 100 000 palabras. Aunque algunos LLM aceptan contextos largos,
meter el documento entero es caro, lento y **empeora la precisión** (el modelo "se pierde" en el medio).
La práctica estándar es:

1. Dividir el texto en **oraciones** (para medir tono oración por oración).
2. Agrupar oraciones en **chunks** de ~150 palabras con un pequeño **solape** (para no cortar una idea a
   la mitad). Los chunks son la unidad que el sistema RAG va a buscar y entregarle al LLM.

> ¿Por qué 150 palabras? El modelo de embeddings que usaremos (`all-MiniLM-L6-v2`) trunca a 256 *tokens*
> (≈ 180 palabras en inglés). Un chunk más largo se cortaría sin avisar.

In [ ]:
ABREVIATURAS = r"\b(U\.S|Inc|Corp|Co|Ltd|No|approx|e\.g|i\.e|vs|Mr|Ms|Dr)\."


def dividir_oraciones(texto, min_palabras=4):
    protegido = re.sub(ABREVIATURAS, lambda m: m.group(0).replace(".", "§"), normalizar_espacios(texto))
    partes = re.split(r"(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÑ¿¡\"(\[])", protegido)
    return [p.replace("§", ".").strip() for p in partes if len(p.split()) >= min_palabras]


def crear_chunks(oraciones, max_palabras=150, solape=1):
    chunks, actual = [], []
    for o in oraciones:
        if actual and sum(len(x.split()) for x in actual) + len(o.split()) > max_palabras:
            chunks.append(" ".join(actual))
            actual = actual[-solape:] if solape else []
        actual.append(o)
    if actual:
        chunks.append(" ".join(actual))
    return chunks


filas = []
for anio in ANIOS:
    for seccion, texto in TEXTOS[anio].items():
        for o in dividir_oraciones(texto):
            filas.append({"anio": anio, "seccion": seccion, "oracion": o})
df_oraciones = pd.DataFrame(filas)

print(df_oraciones.groupby(["anio", "seccion"]).size().rename("n_oraciones"))
df_oraciones.head(5)

In [ ]:
# Chunks del reporte más reciente: la base de conocimiento del RAG
filas = []
for seccion in ["1A", "7"]:
    oraciones = df_oraciones.query("anio == @ANIO_ACTUAL and seccion == @seccion")["oracion"].tolist()
    for texto in crear_chunks(oraciones):
        filas.append({"seccion": seccion, "texto": texto})
df_chunks = pd.DataFrame(filas)
df_chunks.insert(0, "id", [f"C{i}" for i in range(len(df_chunks))])

print(f"{len(df_chunks)} chunks | palabras por chunk: "
      f"{df_chunks['texto'].str.split().str.len().describe()[['mean', 'max']].round(0).to_dict()}")
df_chunks.head(3)

<a id="sec3"></a>
## 3. ¿Cambió el tono? ¿Qué riesgos son nuevos?

### 3.1 Tono por sección con FinBERT

Reutilizamos **FinBERT** de la sesión 7, ahora sobre cada oración de cada sección. Un indicador simple y
muy usado es el **tono neto**:

$$\text{tono neto} = \frac{\#\text{positivas} - \#\text{negativas}}{\#\text{oraciones}}$$

Ojo con la interpretación: *Risk Factors* **siempre** es negativo (por diseño legal: la empresa describe
lo que podría salir mal). Lo informativo no es su nivel sino **su cambio entre años**.

In [ ]:
from transformers import pipeline

clasificador_finbert = pipeline("text-classification", model="ProsusAI/finbert",
                                device=0 if DEVICE == "cuda" else -1)

In [ ]:
salidas = clasificador_finbert(df_oraciones["oracion"].tolist(), batch_size=32, truncation=True)
df_oraciones["sentimiento"] = [s["label"] for s in salidas]
df_oraciones["confianza"] = [s["score"] for s in salidas]

tono = (
    df_oraciones.assign(pos=df_oraciones["sentimiento"].eq("positive"),
                        neg=df_oraciones["sentimiento"].eq("negative"))
    .groupby(["seccion", "anio"])
    .agg(n=("oracion", "size"), pct_positivas=("pos", "mean"), pct_negativas=("neg", "mean"))
)
tono["tono_neto"] = tono["pct_positivas"] - tono["pct_negativas"]
display(tono.round(3))

ax = tono["tono_neto"].unstack("anio").plot.bar(figsize=(6, 4), rot=0)
ax.axhline(0, color="black", lw=0.8)
ax.set_title(f"Tono neto por sección — {EMPRESA}")
ax.set_ylabel("tono neto")
plt.show()

In [ ]:
# Las oraciones más negativas del MD&A más reciente: ¿qué está preocupando a la gerencia?
(df_oraciones.query("anio == @ANIO_ACTUAL and seccion == '7' and sentimiento == 'negative'")
 .sort_values("confianza", ascending=False)[["oracion", "confianza"]].head(5))

### 3.2 *Lazy Prices*: el valor del cambio

Cohen, Malloy y Nguyen (2020, *Journal of Finance*) mostraron que las empresas cambian poco su 10-K de un
año a otro, y que **cuando lo cambian** (sobre todo en *Risk Factors*) suelen venir malas noticias que el
mercado tarda meses en incorporar. Medimos el cambio de dos formas:

- **Similitud global** (TF-IDF + coseno) entre la sección de este año y la del anterior: 1 = idéntica.
- **Oraciones nuevas**: para cada oración de este año buscamos la oración **más parecida** del año
  anterior usando embeddings. Si ni la más parecida se le acerca (similitud < `UMBRAL_NOVEDAD`), la
  oración es **nueva**. Esto es exactamente lo que haría un analista con un "comparar documentos", pero
  robusto a cambios menores de redacción.

In [ ]:
from sentence_transformers import SentenceTransformer

modelo_embeddings = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

In [ ]:
def similitud_tfidf(texto_a, texto_b):
    m = TfidfVectorizer(stop_words="english").fit_transform([texto_a, texto_b])
    return float(cosine_similarity(m[0], m[1])[0, 0])


similitudes = pd.Series(
    {s: similitud_tfidf(TEXTOS[ANIO_PREVIO][s], TEXTOS[ANIO_ACTUAL][s]) for s in ["1A", "7"]},
    name=f"similitud {ANIO_PREVIO}→{ANIO_ACTUAL}",
)
similitudes.round(3)

In [ ]:
UMBRAL_NOVEDAD = 0.75   # parámetro de juicio: súbelo y aparecerán más oraciones "nuevas"


def comparar_oraciones(seccion):
    act = df_oraciones.query("anio == @ANIO_ACTUAL and seccion == @seccion").reset_index(drop=True)
    prev = df_oraciones.query("anio == @ANIO_PREVIO and seccion == @seccion").reset_index(drop=True)
    e_act = modelo_embeddings.encode(act["oracion"].tolist(), normalize_embeddings=True)
    e_prev = modelo_embeddings.encode(prev["oracion"].tolist(), normalize_embeddings=True)
    sim = e_act @ e_prev.T
    act["similitud_max"] = sim.max(axis=1)
    prev["similitud_max"] = sim.max(axis=0)
    nuevas = act[act["similitud_max"] < UMBRAL_NOVEDAD]
    eliminadas = prev[prev["similitud_max"] < UMBRAL_NOVEDAD]
    return act, nuevas, eliminadas


act_1a, nuevas_1a, eliminadas_1a = comparar_oraciones("1A")
PROP_NUEVAS_1A = len(nuevas_1a) / len(act_1a)
print(f"Risk Factors {ANIO_ACTUAL}: {len(nuevas_1a)} de {len(act_1a)} oraciones son nuevas "
      f"({PROP_NUEVAS_1A:.0%}); {len(eliminadas_1a)} oraciones de {ANIO_PREVIO} desaparecieron.\n")
display(Markdown("**Oraciones nuevas en Risk Factors:**"))
display(nuevas_1a[["oracion", "sentimiento", "similitud_max"]].sort_values("similitud_max").head(10))

**Para discutir:** ¿las oraciones nuevas son riesgos *genuinamente nuevos* o solo redacción distinta?
Revisa las que quedaron cerca del umbral. Este número (`PROP_NUEVAS_1A`) lo usaremos más adelante como
**una** señal para la prima de riesgo del DCF, de forma explícita y discutible.

<a id="sec4"></a>
## 4. RAG: preguntarle al reporte (y exigir citas)

En la sesión 7 vimos que un LLM **inventa** cuando responde de memoria, y que es mucho más confiable
cuando **extrae de un texto que le damos**. RAG (*Retrieval-Augmented Generation*) automatiza esa buena
práctica:

1. **Indexar**: calcular el embedding de cada chunk del reporte (una sola vez).
2. **Recuperar**: dada una pregunta, buscar los *k* chunks más parecidos (similitud coseno).
3. **Generar**: darle al LLM **solo esos chunks**, cada uno con su id `[C#]`, y pedirle que responda
   citando los ids — o que diga que **no está**.
4. **Verificar**: comprobar en Python que las citas existen y que cada número de la respuesta aparece
   en los fragmentos citados.

> Los embeddings de `all-MiniLM-L6-v2` están entrenados en inglés, así que escribimos la **consulta de
> búsqueda en inglés** (como el 10-K), aunque la pregunta al LLM y su respuesta sean en español.

In [ ]:
E_CHUNKS = modelo_embeddings.encode(df_chunks["texto"].tolist(), normalize_embeddings=True)


def recuperar(consulta, k=4, seccion=None):
    q = modelo_embeddings.encode([consulta], normalize_embeddings=True)[0]
    df = df_chunks.assign(score=E_CHUNKS @ q)
    if seccion:
        df = df[df["seccion"] == seccion]
    return df.sort_values("score", ascending=False).head(k)


recuperar("expected net sales growth outlook next fiscal year", k=3)[["id", "seccion", "score", "texto"]]

In [ ]:
PROMPT_RAG = (
    "Eres un analista financiero. Responde ÚNICAMENTE con base en los fragmentos del reporte anual que "
    "se te entregan, cada uno identificado como [C#]. No uses conocimiento externo. "
    "Si la respuesta no está en los fragmentos, dilo y marca encontrado como false.\n"
    "Responde SOLO con un objeto JSON válido, sin texto adicional, con estas claves:\n"
    '  "respuesta": respuesta breve en español (máximo 3 frases)\n'
    '  "citas": lista con los ids de los fragmentos usados, por ejemplo ["C3", "C5"]\n'
    '  "encontrado": true o false'
)


def formatear_contexto(df_ctx):
    return "\n\n".join(f"[{r.id}] {r.texto}" for r in df_ctx.itertuples())


def _candidatos_numero(token):
    """'5,610' → {5610}; '5,5' → {55, 5.5}; '5.610' → {5.61, 5610}: cubre formato inglés y español."""
    token = token.strip(".,")
    candidatos = set()
    for s in (token.replace(",", ""), token.replace(".", "").replace(",", ".")):
        try:
            candidatos.add(round(float(s), 4))
        except ValueError:
            pass
    return candidatos


def numeros_en(texto):
    candidatos = set()
    for token in re.findall(r"(?<![A-Za-z\d])\d[\d.,]*", texto):
        candidatos |= _candidatos_numero(token)
    return candidatos


def numeros_no_soportados(texto_respuesta, texto_fuente):
    fuente = numeros_en(texto_fuente)
    return [t.strip(".,") for t in re.findall(r"(?<![A-Za-z\d])\d[\d.,]*", texto_respuesta)
            if not (_candidatos_numero(t) & fuente)]


def preguntar_reporte(pregunta, consulta, k=4, seccion=None):
    ctx = recuperar(consulta, k=k, seccion=seccion)
    mensajes = [{"role": "system", "content": PROMPT_RAG},
                {"role": "user", "content": f"Fragmentos:\n{formatear_contexto(ctx)}\n\nPregunta: {pregunta}"}]
    datos = extraer_json(llamar_llm(mensajes, max_new_tokens=300)) or {}
    citas = [c for c in datos.get("citas", []) if c in set(ctx["id"])]
    texto_citado = " ".join(ctx.loc[ctx["id"].isin(citas), "texto"]) if citas else ""
    respuesta = str(datos.get("respuesta", ""))
    return {
        "pregunta": pregunta,
        "respuesta": respuesta,
        "citas": citas,
        "encontrado": datos.get("encontrado"),
        "citas_invalidas": [c for c in datos.get("citas", []) if c not in set(ctx["id"])],
        "numeros_sin_soporte": numeros_no_soportados(respuesta, texto_citado),
        "json_valido": bool(datos),
    }

In [ ]:
PREGUNTAS = [
    ("¿Qué crecimiento de ingresos espera la gerencia para el próximo año fiscal?",
     "outlook expected net sales revenue growth next fiscal year"),
    ("¿Qué explica el cambio en el margen bruto del último año?",
     "gross margin decreased increased basis points due to"),
    ("¿Cuánto espera invertir la empresa en capex el próximo año?",
     "expected capital expenditures next fiscal year approximately"),
    ("¿Cuáles son los principales riesgos para la rentabilidad?",
     "risks that could adversely affect our profitability and gross margin"),
    # Pregunta trampa: la respuesta NO está en un 10-K. Un buen sistema debe decir que no la encontró.
    ("¿Cuál es el precio objetivo de la acción según la gerencia?",
     "target price of our common stock"),
]

df_rag = pd.DataFrame([preguntar_reporte(p, c) for p, c in PREGUNTAS])
df_rag

Revisa con lupa tres columnas:

- `citas_invalidas`: ids que el modelo **inventó** (no estaban en el contexto que le dimos).
- `numeros_sin_soporte`: números de la respuesta que **no aparecen** en los fragmentos citados → posible
  alucinación o cálculo no pedido. (Puede haber falsos positivos, p. ej. si el modelo convierte
  "180 basis points" en "1,8 puntos"; por eso es una alerta para revisar, no un veredicto.)
- La **pregunta trampa**: ¿el modelo admitió que no estaba o se inventó un precio objetivo?

<a id="sec5"></a>
## 5. Extracción estructurada de insumos de valoración… y su verificación

Ahora le pedimos al LLM lo que pediríamos a un analista junior: "sácame del reporte los ingresos, la
utilidad operativa, el capex, la depreciación y la tasa de impuestos del último año". Tres reglas de
diseño que vienen de la sesión 7:

1. **Un campo por llamada**, con su propia búsqueda RAG: con modelos pequeños, pedir todo a la vez
   multiplica los errores.
2. El LLM devuelve el valor **y la cita literal** de donde lo sacó, más la **unidad** tal como aparece.
   La conversión de unidades la hace Python, no el LLM.
3. **Verificamos** cada cifra contra XBRL y comprobamos que la cita exista de verdad en el reporte.

In [ ]:
CAMPOS = {
    "ingresos": ("net sales revenue increased decreased fiscal year million",
                 "los ingresos netos TOTALES de la empresa (net sales / total revenue), no los de un segmento"),
    "utilidad_operativa": ("operating income increased decreased million operating margin",
                           "la utilidad operativa total (operating income)"),
    "dep_amort": ("depreciation and amortization expense million",
                  "el gasto de depreciación y amortización (depreciation and amortization)"),
    "capex": ("capital expenditures were million",
              "el capex EJECUTADO en el año (capital expenditures), no el proyectado para el año siguiente"),
    "tasa_impuestos": ("effective tax rate",
                       "la tasa efectiva de impuestos (effective tax rate), en porcentaje"),
}

FACTOR_UNIDAD = {"millones": 1, "miles de millones": 1000, "miles": 0.001, "porcentaje": 1}

PROMPT_EXTRACCION = (
    "Eres un analista que extrae datos de reportes financieros. Usa SOLO los fragmentos entregados. "
    "No calcules, no estimes y no conviertas unidades.\n"
    "Responde SOLO con un objeto JSON válido con estas claves:\n"
    '  "valor": el número tal como aparece en el texto (sin símbolos ni comas), o null si no aparece\n'
    '  "unidad": una de "millones", "miles de millones", "miles" o "porcentaje"\n'
    '  "cita": la oración del fragmento que contiene el dato, copiada literalmente'
)


def cita_existe(cita, texto_fuente, umbral=0.9):
    """True si la cita aparece literalmente (o casi: ratio ≥ umbral) en alguna oración de la fuente."""
    cita = normalizar_espacios(cita).lower()
    if len(cita) < 15:
        return False
    if cita in normalizar_espacios(texto_fuente).lower():
        return True
    return max((difflib.SequenceMatcher(None, cita, o.lower()).ratio()
                for o in dividir_oraciones(texto_fuente)), default=0) >= umbral


def extraer_campo(campo, anio):
    consulta, descripcion = CAMPOS[campo]
    ctx = recuperar(consulta, k=4, seccion="7")
    pregunta = f"Extrae {descripcion} correspondiente al año fiscal {anio}."
    mensajes = [{"role": "system", "content": PROMPT_EXTRACCION},
                {"role": "user", "content": f"Fragmentos:\n{formatear_contexto(ctx)}\n\n{pregunta}"}]
    datos = extraer_json(llamar_llm(mensajes, max_new_tokens=200)) or {}
    valor = datos.get("valor")
    try:
        valor = float(str(valor).replace(",", "")) * FACTOR_UNIDAD.get(datos.get("unidad"), 1)
    except (TypeError, ValueError):
        valor = None
    return {"campo": campo, "valor_llm": valor, "unidad_llm": datos.get("unidad"),
            "cita": datos.get("cita", ""), "cita_verificada": cita_existe(datos.get("cita", ""), TEXTOS[ANIO_ACTUAL]["7"])}


df_extraccion = pd.DataFrame([extraer_campo(c, ANIO_ACTUAL) for c in CAMPOS])
df_extraccion

In [ ]:
# Verdad de referencia: XBRL
ultimo = FINANCIEROS.loc[ANIO_ACTUAL] if ANIO_ACTUAL in FINANCIEROS.index else FINANCIEROS.iloc[-1]
verdad = ultimo.to_dict()
if pd.notna(verdad.get("impuestos")) and pd.notna(verdad.get("utilidad_antes_impuestos")):
    verdad["tasa_impuestos"] = 100 * verdad["impuestos"] / verdad["utilidad_antes_impuestos"]


def estado(fila):
    real = verdad.get(fila["campo"])
    if fila["valor_llm"] is None:
        return "— no encontrado"
    if real is None or pd.isna(real):
        return "sin dato XBRL"
    return "✅ coincide" if abs(fila["valor_llm"] / real - 1) < 0.01 else "⚠️ difiere"


df_extraccion["valor_xbrl"] = df_extraccion["campo"].map(verdad)
df_extraccion["error_pct"] = 100 * (df_extraccion["valor_llm"] / df_extraccion["valor_xbrl"] - 1)
df_extraccion["estado"] = df_extraccion.apply(estado, axis=1)
df_extraccion[["campo", "valor_llm", "valor_xbrl", "error_pct", "cita_verificada", "estado"]].round(2)

**Lectura de resultados.** Errores típicos que vas a encontrar (sobre todo con el modelo de 3B):

- Tomar la cifra de **un segmento** (p. ej. ventas *direct-to-consumer*) en vez del total.
- Tomar el capex **proyectado** del *Outlook* en lugar del ejecutado.
- Tomar la cifra del **año anterior** que aparece en la misma oración ("…from $5,320 million").
- En reportes reales: la cifra simplemente **no está** en el MD&A (D&A suele estar solo en el estado de
  flujos de caja), o difiere de XBRL por redondeo ("$391.0 billion").

**Conclusión para la práctica profesional:** el LLM es excelente para *encontrar y explicar* dónde está
la información; las **cifras del modelo de valoración salen de la fuente estructurada (XBRL)**. Por eso,
en la siguiente sección el DCF usa XBRL para los números y el LLM solo para la lectura cualitativa.

<a id="sec6"></a>
## 6. Aplicación: un DCF por escenarios informado por el texto

Ya conocen el DCF de *Valoración de empresas*. Lo nuevo aquí es **de dónde salen los supuestos** y
**quién decide qué**:

| Elemento | Fuente | Quién decide |
|---|---|---|
| Cifras históricas (ingresos, EBIT, D&A, capex, caja, deuda, acciones) | XBRL | Dato |
| Guía de crecimiento de la gerencia, presión sobre márgenes, riesgos clave | Texto, leído por el LLM **con citas verificadas** | LLM (verificable) |
| Cómo se traduce esa lectura en números del DCF | Tabla `REGLAS` | **Analista** (explícito y discutible) |
| WACC base y crecimiento terminal | Curso de valoración | **Analista** |

### 6.1 Históricos

In [ ]:
hist = FINANCIEROS.copy()
hist["crecimiento"] = hist["ingresos"].pct_change()
hist["margen_operativo"] = hist["utilidad_operativa"] / hist["ingresos"]
hist["tasa_impuestos"] = hist["impuestos"] / hist["utilidad_antes_impuestos"]
hist["da_pct_ventas"] = hist["dep_amort"] / hist["ingresos"]
hist["capex_pct_ventas"] = hist["capex"] / hist["ingresos"]
hist[["ingresos", "crecimiento", "margen_operativo", "tasa_impuestos", "da_pct_ventas", "capex_pct_ventas"]].round(4)

### 6.2 Lectura cualitativa del LLM

Le pedimos al LLM una lectura de analista: la **guía de crecimiento** de la gerencia (y el rango numérico
que implica, p. ej. *"low-to-mid single digits"* ≈ 2 % a 5 %), la **presión sobre márgenes** y los
**riesgos clave**, **cada uno con su cita literal**. Si una cita no se puede verificar en el reporte,
descartamos ese elemento.

In [ ]:
PROMPT_LECTURA = (
    "Eres un analista de valoración de empresas. Lee los fragmentos del reporte anual y responde SOLO con "
    "un objeto JSON válido con estas claves:\n"
    '  "guia_crecimiento_cita": oración literal donde la gerencia da su expectativa de crecimiento de '
    'ingresos para el próximo año, o "" si no existe\n'
    '  "crecimiento_min_pct" y "crecimiento_max_pct": rango numérico en % que implica esa guía '
    '(por ejemplo "low-to-mid single digits" equivale aproximadamente a 2 y 5), o null si no hay guía\n'
    '  "presion_margen": "alta", "media" o "baja", según lo que digan los fragmentos sobre márgenes\n'
    '  "presion_margen_cita": oración literal que justifica presion_margen\n'
    '  "riesgos_clave": lista de hasta 3 objetos {"riesgo": resumen breve en español, "cita": oración literal}\n'
    "Usa solo información de los fragmentos."
)

ctx_lectura = pd.concat([
    recuperar("outlook expected net sales growth operating margin next fiscal year", k=3, seccion="7"),
    recuperar("gross margin operating margin decreased pressure costs", k=2, seccion="7"),
    recuperar("could materially adversely affect our profitability", k=3, seccion="1A"),
]).drop_duplicates("id")

mensajes = [{"role": "system", "content": PROMPT_LECTURA},
            {"role": "user", "content": f"Fragmentos:\n{formatear_contexto(ctx_lectura)}"}]
lectura = extraer_json(llamar_llm(mensajes, max_new_tokens=600)) or {}

texto_reporte = " ".join(TEXTOS[ANIO_ACTUAL].values())
guia_ok = cita_existe(lectura.get("guia_crecimiento_cita", ""), texto_reporte)
margen_ok = cita_existe(lectura.get("presion_margen_cita", ""), texto_reporte)
riesgos_ok = [r for r in lectura.get("riesgos_clave", []) if cita_existe(r.get("cita", ""), texto_reporte)]

print(json.dumps(lectura, indent=2, ensure_ascii=False))
print(f"\nCita de guía verificada: {guia_ok} | cita de margen verificada: {margen_ok} | "
      f"riesgos con cita verificada: {len(riesgos_ok)} de {len(lectura.get('riesgos_clave', []))}")

### 6.3 De la lectura a los supuestos: reglas explícitas del analista

Esta celda es **el corazón pedagógico del ejercicio**: aquí se ve, sin cajas negras, cómo una lectura
de texto se convierte en supuestos de valoración. Cada número de `REGLAS` es una decisión del analista
que debe poder defender ante un comité. Si no hay guía verificada, usamos el crecimiento histórico.

- **Crecimiento año 1**: punto medio de la guía (base), extremo inferior −1 pp (pesimista), extremo
  superior (optimista). Converge linealmente al crecimiento terminal en el año 5.
- **Margen operativo**: último margen + ajuste según la presión detectada.
- **WACC**: WACC base + una **prima por cambio en Risk Factors** proporcional a `PROP_NUEVAS_1A`
  (la señal de *Lazy Prices*), con tope.

In [ ]:
REGLAS = {
    "wacc_base": 0.090,
    "g_terminal": 0.025,
    "nwc_pct_delta_ventas": 0.10,          # inversión en capital de trabajo = 10% del aumento en ventas
    "ajuste_margen": {"alta": -0.005, "media": -0.0025, "baja": 0.0},
    "prima_max_riesgos_nuevos": 0.005,     # hasta +50 pb de WACC…
    "prop_nuevas_para_prima_max": 0.20,    # …si ≥ 20% de las oraciones de Risk Factors son nuevas
    "escenarios": {   # (ajuste crecimiento, ajuste margen extra, ajuste WACC)
        "pesimista": (-0.01, -0.010, +0.010),
        "base":      (0.00,  0.000,  0.000),
        "optimista": (0.00,  None,  -0.005),   # None = sin presión de margen (vuelve al último margen)
    },
}

u = hist.iloc[-1]
g_hist = float(u["crecimiento"])
if guia_ok and lectura.get("crecimiento_min_pct") is not None and lectura.get("crecimiento_max_pct") is not None:
    g_min, g_max = lectura["crecimiento_min_pct"] / 100, lectura["crecimiento_max_pct"] / 100
    origen_g = "guía de la gerencia (verificada)"
else:
    g_min, g_max = g_hist - 0.01, g_hist + 0.01
    origen_g = "crecimiento histórico ±1 pp (no hubo guía verificada)"

presion = lectura.get("presion_margen") if margen_ok else "media"
prima_texto = REGLAS["prima_max_riesgos_nuevos"] * min(1, PROP_NUEVAS_1A / REGLAS["prop_nuevas_para_prima_max"])

supuestos = {}
for nombre, (aj_g, aj_m, aj_w) in REGLAS["escenarios"].items():
    g1 = {"pesimista": g_min + aj_g, "base": (g_min + g_max) / 2, "optimista": g_max}[nombre]
    margen = u["margen_operativo"] + (0 if aj_m is None else REGLAS["ajuste_margen"].get(presion, 0) + aj_m)
    supuestos[nombre] = {
        "crecimiento_anio1": g1,
        "margen_operativo": margen,
        "wacc": REGLAS["wacc_base"] + prima_texto + aj_w,
    }

print(f"Origen del crecimiento: {origen_g} | presión de margen: {presion} | "
      f"prima por riesgos nuevos: {prima_texto * 1e4:.0f} pb")
pd.DataFrame(supuestos).round(4)

### 6.4 El DCF

Flujo de caja libre de la firma: $FCFF_t = EBIT_t(1-t) + D\&A_t - Capex_t - \Delta NWC_t$, cinco años
explícitos y valor terminal de Gordon. D&A y capex como % de ventas = promedio histórico; tasa de
impuestos = promedio histórico XBRL.

In [ ]:
def dcf(ingresos_0, crecimiento_anio1, margen_operativo, wacc, g_terminal, tasa_impuestos,
        da_pct, capex_pct, nwc_pct, caja, deuda, acciones, anios=5):
    crecimientos = np.linspace(crecimiento_anio1, g_terminal, anios)
    filas, ingresos_prev = [], ingresos_0
    for t, g in enumerate(crecimientos, start=1):
        ingresos = ingresos_prev * (1 + g)
        ebit = ingresos * margen_operativo
        fcff = ebit * (1 - tasa_impuestos) + ingresos * da_pct - ingresos * capex_pct - nwc_pct * (ingresos - ingresos_prev)
        filas.append({"anio": t, "crecimiento": g, "ingresos": ingresos, "ebit": ebit, "fcff": fcff,
                      "factor_desc": 1 / (1 + wacc) ** t})
        ingresos_prev = ingresos
    proy = pd.DataFrame(filas).set_index("anio")
    vp_flujos = (proy["fcff"] * proy["factor_desc"]).sum()
    valor_terminal = proy["fcff"].iloc[-1] * (1 + g_terminal) / (wacc - g_terminal)
    vp_terminal = valor_terminal * proy["factor_desc"].iloc[-1]
    valor_firma = vp_flujos + vp_terminal
    patrimonio = valor_firma + caja - deuda
    return {"valor_firma": valor_firma, "vp_terminal_pct": vp_terminal / valor_firma,
            "patrimonio": patrimonio, "valor_por_accion": patrimonio / acciones}, proy


PARAMS_COMUNES = dict(
    ingresos_0=u["ingresos"], g_terminal=REGLAS["g_terminal"],
    tasa_impuestos=hist["tasa_impuestos"].mean(), da_pct=hist["da_pct_ventas"].mean(),
    capex_pct=hist["capex_pct_ventas"].mean(), nwc_pct=REGLAS["nwc_pct_delta_ventas"],
    caja=u["caja"], deuda=u["deuda"], acciones=u["acciones"],
)

resultados, proyecciones = {}, {}
for nombre, s in supuestos.items():
    resultados[nombre], proyecciones[nombre] = dcf(**PARAMS_COMUNES, **s)

df_valor = pd.DataFrame(resultados).T
df_valor["vs_precio_mercado"] = df_valor["valor_por_accion"] / PRECIO_MERCADO - 1 if PRECIO_MERCADO else np.nan
display(df_valor.round(3))
display(Markdown("**Proyección del escenario base (millones):**"))
proyecciones["base"].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (a) Rango de valor por escenario vs precio de mercado
orden = ["pesimista", "base", "optimista"]
axes[0].barh(orden, df_valor.loc[orden, "valor_por_accion"], color=["#c0504d", "#7f7f7f", "#4f81bd"])
if PRECIO_MERCADO:
    axes[0].axvline(PRECIO_MERCADO, color="black", ls="--", label=f"Precio de mercado: {PRECIO_MERCADO:.2f}")
    axes[0].legend(loc="lower right")
for i, v in enumerate(df_valor.loc[orden, "valor_por_accion"]):
    axes[0].text(v, i, f" {v:.2f}", va="center")
axes[0].set_title("Valor por acción por escenario")
axes[0].set_xlabel("USD por acción")

# (b) Sensibilidad del escenario base a WACC y g
waccs = np.round(supuestos["base"]["wacc"] + np.arange(-0.02, 0.0201, 0.01), 4)
gs = np.round(np.arange(0.015, 0.0351, 0.005), 4)
tabla = pd.DataFrame(
    [[dcf(**{**PARAMS_COMUNES, "g_terminal": g},
          crecimiento_anio1=supuestos["base"]["crecimiento_anio1"],
          margen_operativo=supuestos["base"]["margen_operativo"], wacc=w)[0]["valor_por_accion"]
      for g in gs] for w in waccs],
    index=[f"{w:.1%}" for w in waccs], columns=[f"{g:.1%}" for g in gs],
)
sns.heatmap(tabla, annot=True, fmt=".1f", cmap="RdBu", center=PRECIO_MERCADO, ax=axes[1], cbar=False)
axes[1].set_title("Sensibilidad (base): valor por acción")
axes[1].set_xlabel("crecimiento terminal")
axes[1].set_ylabel("WACC")
plt.tight_layout()
plt.show()

### 6.5 Nota de valoración asistida por IA (insumo del Taller 2)

Por último, el LLM **redacta** una nota corta a partir de un resumen estructurado de los resultados.
Le prohibimos usar cifras que no estén en ese resumen y lo **comprobamos** con la misma función de
números de la sección 4. El LLM redacta; el analista firma.

In [ ]:
resumen_valoracion = {
    "empresa": EMPRESA,
    "anio_fiscal": int(ANIO_ACTUAL),
    "precio_mercado": None if PRECIO_MERCADO is None else round(PRECIO_MERCADO, 2),
    "valor_por_accion": {k: round(float(v), 2) for k, v in df_valor["valor_por_accion"].items()},
    "supuestos": {k: {kk: f"{vv:.1%}" for kk, vv in v.items()} for k, v in supuestos.items()},
    "crecimiento_ultimo_anio": f"{g_hist:.1%}",
    "margen_operativo_ultimo_anio": f"{u['margen_operativo']:.1%}",
    "origen_crecimiento": origen_g,
    "guia_gerencia": lectura.get("guia_crecimiento_cita", "") if guia_ok else "",
    "presion_margen": presion,
    "riesgos_clave": [r["riesgo"] for r in riesgos_ok],
    "proporcion_oraciones_nuevas_risk_factors": f"{PROP_NUEVAS_1A:.0%}",
}

PROMPT_NOTA = (
    "Eres un analista de valoración. Redacta en español una nota de valoración de máximo 200 palabras con "
    "tres párrafos: (1) desempeño reciente y guía de la gerencia, (2) riesgos y cómo se reflejan en los "
    "escenarios, (3) comparación del rango de valor con el precio de mercado. "
    "Usa EXCLUSIVAMENTE las cifras del JSON que se te entrega; no agregues ninguna otra cifra. "
    "No hagas recomendaciones de compra o venta: presenta la conclusión como análisis."
)

nota = llamar_llm([{"role": "system", "content": PROMPT_NOTA},
                   {"role": "user", "content": json.dumps(resumen_valoracion, ensure_ascii=False)}],
                  max_new_tokens=500)
display(Markdown(nota))

sin_soporte = numeros_no_soportados(nota, json.dumps(resumen_valoracion, ensure_ascii=False))
print("\nCifras de la nota que NO están en los datos de entrada:", sin_soporte or "ninguna ✅")

### 6.6 Discusión

- **¿Qué aportó la IA?** Velocidad (leer cientos de páginas en minutos), consistencia (la misma lectura
  para 50 empresas) y **trazabilidad** (cada supuesto cualitativo tiene una cita verificable).
- **¿Qué no aportó?** Ninguna cifra del DCF salió del LLM, y la traducción "texto → supuesto" está en
  `REGLAS`, a la vista. Si un comité pregunta "¿por qué el WACC pesimista es 10,3 %?", la respuesta está
  en una celda, no en una caja negra.
- **Riesgos:** sesgo de la gerencia (la guía es optimista por diseño: compara la guía de hace un año con
  lo realizado), lenguaje *boilerplate* en *Risk Factors*, errores de extracción y alucinaciones.
- **Ética y regulación:** una nota de valoración asistida por IA sigue siendo responsabilidad del
  analista; documente siempre modelo, versión, prompts y verificaciones (reproducibilidad).

<a id="sec7"></a>
## 7. Ejercicios

### Ejercicio 1 (introductorio) — ¿Cumplió la gerencia?
Con el caso ficticio, usa `preguntar_reporte` sobre el reporte de **2024** para extraer la guía de
crecimiento que la gerencia dio para 2025 y compárala con el crecimiento realizado en XBRL. Repite con
tu empresa real usando dos 10-K consecutivos. ¿Qué implica para el peso que le das a la guía en tu DCF?

*Pista: reconstruye `df_chunks` y `E_CHUNKS` con `ANIO_PREVIO` en lugar de `ANIO_ACTUAL`.*

### Ejercicio 2 (intermedio) — Un reporte colombiano en PDF
Descarga el informe de gestión o el informe periódico de fin de ejercicio de un emisor colombiano (por
ejemplo, desde la página de relación con inversionistas de la empresa) y súbelo a Colab. Adapta el flujo:

```python
from pypdf import PdfReader
texto_pdf = " ".join(p.extract_text() or "" for p in PdfReader("informe.pdf").pages)
oraciones = dividir_oraciones(texto_pdf)
```

Cambia el modelo de embeddings por uno **multilingüe** (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`)
y escribe las consultas en español. FinBERT no sirve en español: ¿qué usarías para medir el tono?

### Ejercicio 3 (retador) — *Lazy Prices* en pequeño
Para 10 empresas de un mismo sector, calcula la similitud TF-IDF de *Risk Factors* entre los dos últimos
10-K y el retorno de la acción en los 60 días hábiles posteriores a la publicación del último. ¿Las de
menor similitud tuvieron peor desempeño? Con 10 empresas no hay significancia estadística: discute qué
haría falta para un test serio (muestra, horizonte, ajuste por riesgo).

## 8. Taller 2 (evaluable) — Nota de valoración asistida por IA

Para la empresa de tu proyecto integrador entrega este cuaderno ejecutado con:

1. Las secciones 1A y 7 de los dos últimos 10-K (o el equivalente colombiano), con el análisis de tono y
   las oraciones nuevas comentadas.
2. La tabla de verificación de la sección 5 y un párrafo sobre los errores del LLM que encontraste.
3. Tu propia versión de `REGLAS`, **justificada**: por qué esos ajustes y no otros.
4. La nota de valoración final, **editada y firmada por ti**, señalando qué cambiaste del borrador de la IA.